# DX 704 Week 8 Project

This homework will modify a simulator controlling a small vehicle to implement tabular q-learning.
You will first test your code with random and greedy-epsilon policies, then tweak your own training method for a more optimal policy.

The full project description and a template notebook are available on GitHub: [Project 8 Materials](https://github.com/bu-cds-dx704/dx704-project-08).


## Example Code

You may find it helpful to refer to these GitHub repositories of Jupyter notebooks for example code.

* https://github.com/bu-cds-omds/dx601-examples
* https://github.com/bu-cds-omds/dx602-examples
* https://github.com/bu-cds-omds/dx603-examples
* https://github.com/bu-cds-omds/dx704-examples

Any calculations demonstrated in code examples or videos may be found in these notebooks, and you are allowed to copy this example code in your homework answers.

## Rover Simulator

The following Python class implements a simulation of a simple vehicle with integer x,y coordinates facing in one of 8 possible directions.


In [1]:
# DO NOT CHANGE

import random

class RoverSimulator(object):
    DIRECTIONS = ((0, 1), (1, 1), (1, 0), (1, -1), (0, -1), (-1, -1), (-1, 0), (-1, 1))

    def __init__(self, resolution):
        self.resolution = resolution
        self.terminal_state = self.construct_state(resolution // 2, resolution // 2, 0)

        self.initial_states = []
        for initial_x in (0, resolution // 2, resolution - 1):
            for initial_y in (0, resolution // 2, resolution - 1):
                for initial_direction in range(8):
                    initial_state = self.construct_state(initial_x, initial_y, initial_direction)
                    if initial_state != self.terminal_state:
                        self.initial_states.append(initial_state)

    def construct_state(self, x, y, direction):
        assert 0 <= x < self.resolution
        assert 0 <= y < self.resolution
        assert 0 <= direction < 8

        state = (y * self.resolution + x) * 8 + direction
        assert self.decode_state(state) == (x, y, direction)
        return state

    def decode_state(self, state):
        direction = state % 8
        x = (state // 8) % self.resolution
        y = state // (8 * self.resolution)

        return (x, y, direction)

    def get_actions(self, state):
        return [-1, 0, 1]

    def get_next_reward_state(self, curr_state, curr_action):
        if curr_state == self.terminal_state:
            # no rewards or changes from terminal state
            return (0, curr_state)

        (curr_x, curr_y, curr_direction) = self.decode_state(curr_state)
        (curr_dx, curr_dy) = self.DIRECTIONS[curr_direction]

        assert self.construct_state(curr_x, curr_y, curr_direction) == curr_state

        assert curr_action in (-1, 0, 1)

        next_x = min(max(0, curr_x + curr_dx), self.resolution - 1)
        next_y = min(max(0, curr_y + curr_dy), self.resolution - 1)
        next_direction = (curr_direction + curr_action) % 8

        next_state = self.construct_state(next_x, next_y, next_direction)
        next_reward = 1 if next_state == self.terminal_state else 0

        return (next_reward, next_state)

    def rollout_policy(self, policy_func, max_steps=1000):
        curr_state = self.sample_initial_state()
        for i in range(max_steps):
            curr_action = policy_func(curr_state, self.get_actions(curr_state))
            (next_reward, next_state) = self.get_next_reward_state(curr_state, curr_action)
            yield (curr_state, curr_action, next_reward, next_state)
            curr_state = next_state

    def sample_initial_state(self):
        return random.choice(self.initial_states)

In [2]:
simulator = RoverSimulator(16)
initial_sample = simulator.sample_initial_state()
print("INITIAL SAMPLE", initial_sample)

INITIAL SAMPLE 1091


## Part 1: Implement a Random Policy

Random policies are often used to test simulators and start initial exploration.
Implement a random policy for these simulators.

In [3]:
# YOUR CHANGES HERE

def random_policy(state, actions):
    return random.choice(actions)

Use the code below to test your random policy.
Then modify it to save the results in "log-random.tsv" with the columns curr_state, curr_action, next_reward and next_state.

In [4]:
# YOUR CHANGES HERE

for (curr_state, curr_action, next_reward, next_state) in simulator.rollout_policy(random_policy, max_steps=32):
    print("CURR STATE", curr_state, "ACTION", curr_action, "NEXT REWARD", next_reward, "NEXT STATE", next_state)

CURR STATE 67 ACTION 1 NEXT REWARD 0 NEXT STATE 76
CURR STATE 76 ACTION 1 NEXT REWARD 0 NEXT STATE 77
CURR STATE 77 ACTION 0 NEXT REWARD 0 NEXT STATE 69
CURR STATE 69 ACTION -1 NEXT REWARD 0 NEXT STATE 60
CURR STATE 60 ACTION -1 NEXT REWARD 0 NEXT STATE 59
CURR STATE 59 ACTION -1 NEXT REWARD 0 NEXT STATE 66
CURR STATE 66 ACTION 0 NEXT REWARD 0 NEXT STATE 74
CURR STATE 74 ACTION 1 NEXT REWARD 0 NEXT STATE 83
CURR STATE 83 ACTION 1 NEXT REWARD 0 NEXT STATE 92
CURR STATE 92 ACTION 1 NEXT REWARD 0 NEXT STATE 93
CURR STATE 93 ACTION -1 NEXT REWARD 0 NEXT STATE 84
CURR STATE 84 ACTION -1 NEXT REWARD 0 NEXT STATE 83
CURR STATE 83 ACTION -1 NEXT REWARD 0 NEXT STATE 90
CURR STATE 90 ACTION -1 NEXT REWARD 0 NEXT STATE 97
CURR STATE 97 ACTION 0 NEXT REWARD 0 NEXT STATE 233
CURR STATE 233 ACTION 1 NEXT REWARD 0 NEXT STATE 370
CURR STATE 370 ACTION -1 NEXT REWARD 0 NEXT STATE 377
CURR STATE 377 ACTION 1 NEXT REWARD 0 NEXT STATE 506
CURR STATE 506 ACTION 1 NEXT REWARD 0 NEXT STATE 507
CURR STATE 507

In [5]:
import pandas as pd

state      = []
action     = []
reward     = []
nextState = []

for (curr_state, curr_action, next_reward, next_state) in simulator.rollout_policy(random_policy, max_steps=32):
    state.append(curr_state)
    action.append(curr_action)
    reward.append(next_reward)
    nextState.append(next_state)

df = pd.DataFrame({
    'curr_state'    : state,
    'curr_action'   : action,
    'next_reward'   : reward,
    'next_state'    : nextState
})

In [6]:
df.to_csv('log-random.tsv', sep='\t', index=False)

Submit "log-random.tsv" in Gradescope.

## Part 2: Implement Q-Learning with Random Policy

The code below runs 32 random rollouts of 1024 steps using your random policy.
Modify the rollout code to implement Q-Learning.
Just implement one learning update for each sampled state-action in the simulation.
Use $\alpha=1$ and $\gamma=0.9$ since the simulator is deterministic and there is a sink where the rewards stop.




In [7]:
# YOUR CHANGES HERE

alpha = 1.0
gamma = 0.9

Q = {}

Q_data = []

for episode in range(32):
    for (curr_state, curr_action, next_reward, next_state) in simulator.rollout_policy(random_policy, max_steps=1024):

        # get current/old value from Q table, default to 0.0 if not visited
        old_value = Q.get((curr_state, curr_action), 0.0)

        # max Q-value for all possible actions in next state
        max_next_q = max(Q.get((next_state, a), 0.0) for a in simulator.get_actions(next_state))

        # calculate new Q-value
        new_value = next_reward + gamma * max_next_q

        # update Q table
        Q[(curr_state, curr_action)] = new_value

        Q_data.append({
            'curr_state'    : curr_state,
            'curr_action'   : curr_action,
            'next_reward'   : next_reward,
            'next_state'    : next_state,
            'old_value'     : old_value,
            'new_value'     : new_value
        })

Q_df = pd.DataFrame(Q_data)
Q_df

,curr_state,curr_action,next_reward,next_state,old_value,new_value
0,1095,1,0,1208,0.0,0.0
1,1208,0,0,1336,0.0,0.0
2,1336,1,0,1465,0.0,0.0
3,1465,0,0,1601,0.0,0.0
4,1601,1,0,1738,0.0,0.0
...,...,...,...,...,...,...
32763,1670,1,0,1671,0.0,0.0
32764,1671,0,0,1799,0.0,0.0
32765,1799,-1,0,1926,0.0,0.0
32766,1926,1,0,1927,0.0,0.0


Save each step in the simulator in a file "q-random.tsv" with columns curr_state, curr_action, next_reward, next_state, old_value, new_value.

In [8]:
# YOUR CHANGES HERE

Q_df.to_csv('q-random.tsv', sep='\t', index=False)

Submit "q-random.tsv" in Gradescope.

## Part 3: Implement Epsilon-Greedy Policy

Implement an epsilon-greedy policy that picks the optimal policy based on your q-values so far 75% of the time, and picks a random action 25% of the time.
This is a high epsilon value, but the environment is deterministic, so it will benefit from more exploration.

In [9]:
# YOUR CHANGES HERE

# hard-code epsilon=0.25. this is high but the environment is deterministic.
def epsilon_greedy_policy(state, actions):
    epsilon = 0.25
    if random.random() < epsilon:
        return random.choice(actions)

    # pick best action based on current Q values
    else:
        best_action = max(actions, key=lambda a: Q.get((state, a), 0.0))
        return best_action

Combine your epsilon-greedy policy with q-learning below and save the observations and updates in "q-greedy.tsv" with columns curr_state, curr_action, next_reward, next_state, old_value, new_value.

Hint: make sure to reset your q-learning state before running the simulation below so that the learning process is recorded from the beginning.

In [10]:
alpha = 1.0
gamma = 0.9

Q = {}
Q_data = []

for episode in range(32):
    for (curr_state, curr_action, next_reward, next_state) in simulator.rollout_policy(epsilon_greedy_policy, max_steps=1024):
        
        # get current/old value from Q table, default to 0.0 if not visited
        old_value = Q.get((curr_state, curr_action), 0.0)

        # max Q-value for all possible actions in next state
        max_next_q = max(Q.get((next_state, a), 0.0) for a in simulator.get_actions(next_state))

        # calculate new Q-value
        new_value = next_reward + gamma * max_next_q

        # update Q table
        Q[(curr_state, curr_action)] = new_value

        Q_data.append({
            'curr_state'    : curr_state,
            'curr_action'   : curr_action,
            'next_reward'   : next_reward,
            'next_state'    : next_state,
            'old_value'     : old_value,
            'new_value'     : new_value
        })

        if next_reward > 0:
            # moving to terminal state
            break

Q_df = pd.DataFrame(Q_data)
Q_df

,curr_state,curr_action,next_reward,next_state,old_value,new_value
0,1030,-1,0,1029,0.00,0.000
1,1029,-1,0,900,0.00,0.000
2,900,-1,0,771,0.00,0.000
3,771,-1,0,650,0.00,0.000
4,650,-1,0,657,0.00,0.000
...,...,...,...,...,...,...
13074,449,-1,0,584,0.00,0.000
13075,584,0,0,712,0.00,0.729
13076,712,-1,0,847,0.81,0.810
13077,847,1,0,960,0.90,0.900


In [11]:
Q_df.to_csv('q-greedy.tsv', sep='\t', index=False)

Submit "q-greedy.tsv" in Gradescope.

## Part 4: Extract Policy from Q-Values

Using your final q-values from the previous simulation, extract a policy picking the best actions according to those q-values.
Save the policy in a file "policy-greedy.tsv" with columns state and action.

In [12]:
Q_greedy = pd.read_csv('q-greedy.tsv', sep='\t')

# final Q value per (curr_state, curr_action) pair
final_Q_greedy = Q_greedy.drop_duplicates(subset=['curr_state', 'curr_action'], keep='last')

# max Q values
idx_max = final_Q_greedy.groupby('curr_state')['new_value'].idxmax()
best_actions = final_Q_greedy.loc[idx_max]

best_actions

,curr_state,curr_action,next_reward,next_state,old_value,new_value
9810,2,-1,0,9,0.0,0.0
3155,3,1,0,12,0.0,0.0
12161,4,1,0,5,0.0,0.0
12158,5,1,0,6,0.0,0.0
12791,6,-1,0,5,0.0,0.0
...,...,...,...,...,...,...
7457,2041,-1,0,2040,0.0,0.0
7355,2042,-1,0,2041,0.0,0.0
5443,2043,-1,0,1914,0.0,0.0
11289,2046,-1,0,2037,0.0,0.0


In [13]:
policy_greedy = best_actions[['curr_state', 'curr_action']].copy()
policy_greedy.rename(columns={'curr_state': 'state', 'curr_action': 'action'}, inplace=True)
policy_greedy.to_csv('policy-greedy.tsv', sep='\t', index=False)

Submit "policy-greedy.tsv" in Gradescope.

## Part 5: Implement Large Policy

Train a more optimal policy using q-learning.
Save the policy in a file "policy-optimal.tsv" with columns state and action.

Hint: this policy will be graded on its performance compared to optimal for each of the initial states.
**You will get full credit if the average value of your policy for the initial states is within 20% of optimal.**
Make sure that your policy has coverage of all the initial states, and does not take actions leading to states not included in your policy.
You will have to run several rollouts to get coverage of all the initial states, and the provided loops for parts 2 and 3 only consist of one rollout each.

Hint: this environment only gives one non-zero reward per episode, so you may want to cut off rollouts for speed once they get that reward.
But make sure you update the q-values first!

In [14]:
# YOUR CHANGES HERE

alpha = 1.0
gamma = 0.9

Q = {}
Q_data = []

for episode in range(1000):
    for (curr_state, curr_action, next_reward, next_state) in simulator.rollout_policy(epsilon_greedy_policy, max_steps=1024):
        
        # get current/old value from Q table, default to 0.0 if not visited
        old_value = Q.get((curr_state, curr_action), 0.0)

        # max Q-value for all possible actions in next state
        max_next_q = max(Q.get((next_state, a), 0.0) for a in simulator.get_actions(next_state))

        # calculate new Q-value
        new_value = next_reward + gamma * max_next_q

        # update Q table
        Q[(curr_state, curr_action)] = new_value

        Q_data.append({
            'curr_state'    : curr_state,
            'curr_action'   : curr_action,
            'next_reward'   : next_reward,
            'next_state'    : next_state,
            'old_value'     : old_value,
            'new_value'     : new_value
        })

        if next_reward > 0:
            # moving to terminal state
            break

Q_df = pd.DataFrame(Q_data)

# final Q value per (curr_state, curr_action) pair
final_Q = Q_df.drop_duplicates(subset=['curr_state', 'curr_action'], keep='last')

# max Q values
idx_max = final_Q.groupby('curr_state')['new_value'].idxmax()
best_actions = final_Q.loc[idx_max]

policy_optimal = best_actions[['curr_state', 'curr_action']].rename(
    columns={'curr_state': 'state', 'curr_action': 'action'}
)

policy_optimal

,state,action
57062,0,-1
62463,1,1
64107,2,0
65766,3,-1
65765,4,-1
...,...,...
60349,2043,-1
63827,2044,-1
65647,2045,-1
65297,2046,0


In [15]:
policy_optimal.to_csv('policy-optimal.tsv', sep='\t', index=False)

Submit "policy-optimal.tsv" in Gradescope.

## Part 6: Code

Please submit a Jupyter notebook that can reproduce all your calculations and recreate the previously submitted files.
You do not need to provide code for data collection if you did that by manually.

## Part 7: Acknowledgements

If you discussed this assignment with anyone, please acknowledge them here.
If you did this assignment completely on your own, simply write none below.

If you used any libraries not mentioned in this module's content, please list them with a brief explanation what you used them for. If you did not use any other libraries, simply write none below.

If you used any generative AI tools, please add links to your transcripts below, and any other information that you feel is necessary to comply with the generative AI policy. If you did not use any generative AI tools, simply write none below.